# Advanced Python Exception Handling
## 22 challenging problems with complete worked solutions

**Topic source:** the uploaded “Handling Exceptions” lesson. This notebook builds on its examples of matching handlers, propagation, `else`, `finally`, nested `try`, custom conversion errors, and EAFP versus LBYL. The new exercises and solutions are expansions rather than extracts from the lesson.

**Environment:** Python 3.11+; standard library only. **How to use:** restart the kernel and run all cells in order. Every problem contains a specification, reasoning, complete solution, and executable `assert`-based tests. Tests deliberately provoke exceptions *inside* test helpers so executing the notebook does not stop at a demonstration traceback. Problems 1–21 are independent apart from setup imports; Problem 22 intentionally composes Problems 7 and 14. Identifiers are prefixed or distinctive where helpful.

**Learning path:** precise exception boundaries → propagation and chaining → domain-specific errors → parsing and EAFP → recoverability/retries → guaranteed cleanup and transactions → modern exception APIs → integration capstone.

**Best-practice corrections to remember:** `else` (not `except`) handles success when the `try` suite completes normally; put only the fallible operation in `try` if you do not want the handler to intercept subsequent code. `finally` normally runs when execution leaves `try`, but a killed process or abrupt interpreter exit is not guaranteed to run cleanup. `except Exception` does not catch `KeyboardInterrupt` or `SystemExit`; catch `BaseException` only when re-raising after essential cleanup. EAFP means “easier to ask forgiveness than permission,” not that exceptions should always replace straightforward validation.


In [1]:
import asyncio
import contextlib
import io
import json
import logging
import os
import sqlite3
import tempfile
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

print("Setup OK: standard-library-only exercises. Python 3.11+ is required for Problem 19.")


Setup OK: standard-library-only exercises. Python 3.11+ is required for Problem 19.


---
## Problem 01 · Handler resolution: specific before general

**Difficulty:** Advanced

### Challenge
Write `classify_error(exc)` that raises the supplied exception and returns a label from handlers. Show that the *first compatible handler* wins, and that subclasses of `ValueError` should be handled before their base class. Include an unknown `RuntimeError` case. Do not accidentally swallow `KeyboardInterrupt`.

**Acceptance criteria:** The supplied assertions must pass without requiring network access or third-party libraries. Try solving it before opening the solution.


### Worked solution · reasoning
`except` clauses are examined top-to-bottom. A subclass match is sufficient, so define specialized clauses before generic ones. An explicit `Exception` fallback can turn application exceptions into a category without also catching shutdown signals.


In [2]:
class MalformedField(ValueError):
    """A specific data-validation failure."""


def classify_error(exc: Exception) -> str:
    try:
        raise exc
    except MalformedField:
        return "malformed-field"
    except ValueError:
        return "value"
    except KeyError:
        return "missing-key"
    except Exception:
        return "other-application-error"


def bad_order_demo() -> str:
    try:
        raise MalformedField("wrong shape")
    except Exception:
        return "generic-wins"
    except MalformedField:  # unreachable for this exception
        return "specific-wins"


### Verification · deterministic examples, boundary cases, and regression tests


In [3]:
assert classify_error(MalformedField("x")) == "malformed-field"
assert classify_error(ValueError("x")) == "value"
assert classify_error(KeyError("x")) == "missing-key"
assert classify_error(RuntimeError("x")) == "other-application-error"
assert bad_order_demo() == "generic-wins"
try:
    classify_error(KeyboardInterrupt())
except KeyboardInterrupt:
    pass
else:
    raise AssertionError("KeyboardInterrupt must propagate")
print("Problem 01 passed: handler order and shutdown propagation")


Problem 01 passed: handler order and shutdown propagation


**Review / production note:** Avoid `except:` or `except BaseException:` for general recovery. Order specialized handlers before their parent classes.


---
## Problem 02 · The success path and the narrow try block

**Difficulty:** Advanced

### Challenge
Implement `lookup_and_report(store, key, report)` so a missing key returns `None`, but a `ValueError` raised by the reporting callback propagates. Call the callback **only after successful lookup**. Compare this to an incorrectly broad `try`.

**Acceptance criteria:** The supplied assertions must pass without requiring network access or third-party libraries. Try solving it before opening the solution.


### Worked solution · reasoning
Keep only `store[key]` inside `try`. The `else` suite executes only if the protected operation succeeds. `report` is outside the handler’s scope even though it is in `else`, so an exception from it cannot be confused with a lookup error.


In [4]:
def lookup_and_report(store, key, report):
    try:
        value = store[key]
    except KeyError:
        return None
    else:
        report(value)
        return value


def overly_broad_lookup(store, key, report):
    # Anti-pattern: treats callback failures as if they were key lookup failures.
    try:
        value = store[key]
        report(value)
    except (KeyError, ValueError):
        return None
    return value


### Verification · deterministic examples, boundary cases, and regression tests


In [5]:
events_02 = []
assert lookup_and_report({"ok": 7}, "ok", events_02.append) == 7
assert events_02 == [7]
assert lookup_and_report({}, "missing", events_02.append) is None
assert events_02 == [7]
def broken_report(value):
    raise ValueError("report failed")
try:
    lookup_and_report({"ok": 7}, "ok", broken_report)
except ValueError as exc:
    assert str(exc) == "report failed"
else:
    raise AssertionError("callback error should not be translated")
assert overly_broad_lookup({"ok": 7}, "ok", broken_report) is None
print("Problem 02 passed: else separates a successful lookup from callback errors")


Problem 02 passed: else separates a successful lookup from callback errors


**Review / production note:** An `else` attached to `try` is not interchangeable with an unindented statement after `except`: that later statement also runs after a handled exception unless the handler exits.


---
## Problem 03 · Finally, returns, and exception masking

**Difficulty:** Advanced

### Challenge
Predict the event order of successful and failed operations. Implement a wrapper that closes a resource in `finally` while preserving the operation’s return value or exception. Demonstrate why `return` in `finally` is dangerous, but do not use that pattern in the correct solution.

**Acceptance criteria:** The supplied assertions must pass without requiring network access or third-party libraries. Try solving it before opening the solution.


### Worked solution · reasoning
A `finally` suite normally runs before a `return` finishes and before an uncaught exception propagates. Cleanup code must not `return`, and cleanup failures may mask a primary failure; specialized resource-management policies follow later.


In [6]:
class DemoResource03:
    def __init__(self, events):
        self.events = events
    def close(self):
        self.events.append("close")


def safe_operation03(resource, events, fail=False):
    try:
        events.append("start")
        if fail:
            raise ValueError("original failure")
        events.append("computed")
        return 42
    finally:
        resource.close()


def anti_pattern03():
    try:
        raise RuntimeError("important failure")
    finally:
        return "hidden by finally"  # demonstration only: never do this.


### Verification · deterministic examples, boundary cases, and regression tests


In [7]:
events_03 = []
assert safe_operation03(DemoResource03(events_03), events_03) == 42
assert events_03 == ["start", "computed", "close"]
events_03 = []
try:
    safe_operation03(DemoResource03(events_03), events_03, fail=True)
except ValueError as exc:
    assert str(exc) == "original failure"
else:
    raise AssertionError("original failure must escape")
assert events_03 == ["start", "close"]
assert anti_pattern03() == "hidden by finally"
print("Problem 03 passed: cleanup timing and masking demonstration")


Problem 03 passed: cleanup timing and masking demonstration


**Review / production note:** Use `with` for resources offering context-manager support. `finally` is not a guarantee against process termination, SIGKILL, or a power failure.


---
## Problem 04 · Exception propagation across three call frames

**Difficulty:** Advanced

### Challenge
Create low-, mid-, and top-level functions. The low-level function raises `KeyError`; the middle function logs an event in `finally` without handling the error; the top-level function catches the error, records it, and returns the event log. Show how propagation and cleanup interact.

**Acceptance criteria:** The supplied assertions must pass without requiring network access or third-party libraries. Try solving it before opening the solution.


### Worked solution · reasoning
No matching handler in one frame means the exception continues up the call stack, but that frame’s `finally` executes first. Convert the exception only at the layer that can interpret it.


In [8]:
def low_level04(events):
    events.append("low")
    raise KeyError("token")


def middle04(events):
    try:
        events.append("middle-enter")
        low_level04(events)
    finally:
        events.append("middle-cleanup")


def top_level04():
    events = []
    try:
        middle04(events)
    except KeyError as exc:
        events.append("top-handled:" + exc.args[0])
    return events


### Verification · deterministic examples, boundary cases, and regression tests


In [9]:
assert top_level04() == [
    "middle-enter", "low", "middle-cleanup", "top-handled:token"
]
print("Problem 04 passed: stack unwinding executes cleanup before handling")


Problem 04 passed: stack unwinding executes cleanup before handling


**Review / production note:** `finally` does not itself handle the original error. Log at a meaningful boundary rather than printing and re-raising the same exception in every frame.


---
## Problem 05 · Exception chaining and traceback semantics

**Difficulty:** Advanced

### Challenge
Translate an invalid integer token into a domain-specific `ConfigurationError`. Keep the source exception discoverable through `__cause__`. Contrast explicit `raise ... from exc` with bare `raise` inside a handler; verify original exception identity in the latter case.

**Acceptance criteria:** The supplied assertions must pass without requiring network access or third-party libraries. Try solving it before opening the solution.


### Worked solution · reasoning
Use explicit chaining when changing abstraction level: callers receive a useful domain error and engineers still have the original parse failure. Use bare `raise` to preserve the active exception and traceback when no translation is required.


In [10]:
class ConfigurationError(Exception):
    pass


def parse_port05(token: str) -> int:
    try:
        port = int(token)
    except ValueError as exc:
        raise ConfigurationError(f"invalid port token: {token!r}") from exc
    if not 1 <= port <= 65535:
        raise ConfigurationError("port must be between 1 and 65535")
    return port


def reraised_identity05(original):
    try:
        raise original
    except ValueError:
        raise  # preserve original instance and traceback


### Verification · deterministic examples, boundary cases, and regression tests


In [11]:
assert parse_port05("443") == 443
for bad in ("0", "65536"):
    try:
        parse_port05(bad)
    except ConfigurationError as exc:
        assert exc.__cause__ is None
    else:
        raise AssertionError("out-of-range ports must be rejected")
try:
    parse_port05("forty-three")
except ConfigurationError as exc:
    assert isinstance(exc.__cause__, ValueError)
    assert exc.__suppress_context__ is True
else:
    raise AssertionError("invalid input should be translated")
original_05 = ValueError("same object")
try:
    reraised_identity05(original_05)
except ValueError as exc:
    assert exc is original_05
print("Problem 05 passed: causes and bare re-raise semantics")


Problem 05 passed: causes and bare re-raise semantics


**Review / production note:** Avoid `raise exc` when you mean `raise`: bare `raise` preserves the existing traceback more directly. Exception causes can contain input details; do not leak sensitive data in public messages.


---
## Problem 06 · Design a typed domain-exception hierarchy

**Difficulty:** Advanced

### Challenge
Build a base `ImportError06` plus `MissingColumn06` and `InvalidCell06` subclasses. Make the subclasses carry structured information (`row`, `column`, `value`) rather than forcing callers to parse messages. Implement `require_column06` and `require_positive_int06`.

**Acceptance criteria:** The supplied assertions must pass without requiring network access or third-party libraries. Try solving it before opening the solution.


### Worked solution · reasoning
Exception types form part of a public API. Separate machine-readable fields from the human-readable string, use clear names, and raise the narrowest applicable class. Do not shadow Python’s built-in `ImportError` in application code.


In [12]:
class ImportError06(Exception):
    """Base class for a dataset-import failure (not Python's built-in ImportError)."""


class MissingColumn06(ImportError06):
    def __init__(self, column: str):
        self.column = column
        super().__init__(f"required column missing: {column}")


class InvalidCell06(ImportError06):
    def __init__(self, row: int, column: str, value):
        self.row, self.column, self.value = row, column, value
        super().__init__(f"row {row}, column {column}: invalid value {value!r}")


def require_column06(record, name):
    try:
        return record[name]
    except KeyError as exc:
        raise MissingColumn06(name) from exc


def require_positive_int06(record, row, column):
    raw = require_column06(record, column)
    if isinstance(raw, bool):
        raise InvalidCell06(row, column, raw)
    try:
        number = int(raw)
    except (TypeError, ValueError) as exc:
        raise InvalidCell06(row, column, raw) from exc
    if number <= 0:
        raise InvalidCell06(row, column, raw)
    return number


### Verification · deterministic examples, boundary cases, and regression tests


In [13]:
assert require_positive_int06({"qty": "17"}, 2, "qty") == 17
try:
    require_column06({}, "qty")
except MissingColumn06 as exc:
    assert exc.column == "qty" and isinstance(exc.__cause__, KeyError)
else:
    raise AssertionError("missing column should fail")
for bad in (True, "abc", 0, -1, None):
    try:
        require_positive_int06({"qty": bad}, 9, "qty")
    except InvalidCell06 as exc:
        assert (exc.row, exc.column, exc.value) == (9, "qty", bad)
    else:
        raise AssertionError(f"accepted bad value: {bad!r}")
print("Problem 06 passed: structured domain errors")


Problem 06 passed: structured domain errors


**Review / production note:** A `bool` is an `int` subclass: validate it explicitly when boolean values are not valid integers. Choose input conversion policy deliberately (e.g., `int(2.9)` silently truncates).


---
## Problem 07 · A strict Boolean parser with error contracts

**Difficulty:** Advanced

### Challenge
Implement `strict_bool07` accepting `True`, `False`, integer 0/1, and case-insensitive strings `0/1/t/f/true/false`. Reject floats including 0.0 and 1.0, reject arbitrary strings, and preserve the underlying `ValueError` via chaining for invalid string values. This extends the source lesson’s `make_bool` challenge.

**Acceptance criteria:** The supplied assertions must pass without requiring network access or third-party libraries. Try solving it before opening the solution.


### Worked solution · reasoning
Distinguish unsupported *types* from unsupported *values*. Do not use `bool(token)` for strings: `bool("false")` is `True`. `casefold()` makes documented textual alternatives case-insensitive. Avoid deeply nested exception handlers when simple dispatch is clearer.


In [14]:
class BoolConversionError07(ValueError):
    pass


def strict_bool07(value) -> bool:
    if isinstance(value, bool):
        return value
    if isinstance(value, int):
        if value in (0, 1):
            return bool(value)
        raise BoolConversionError07("integer must be 0 or 1")
    if isinstance(value, str):
        normalized = value.casefold()
        mapping = {"0": False, "f": False, "false": False,
                   "1": True, "t": True, "true": True}
        try:
            return mapping[normalized]
        except KeyError as exc:
            raise BoolConversionError07(f"unsupported Boolean token: {value!r}") from exc
    raise BoolConversionError07(f"unsupported type: {type(value).__name__}")


### Verification · deterministic examples, boundary cases, and regression tests


In [15]:
for token in (True, 1, "1", "T", "tRuE"):
    assert strict_bool07(token) is True
for token in (False, 0, "0", "F", "fAlSe"):
    assert strict_bool07(token) is False
for token in (2, -1, 0.0, 1.0, None, [], "YES", " true "):
    try:
        strict_bool07(token)
    except BoolConversionError07:
        pass
    else:
        raise AssertionError(f"bad token accepted: {token!r}")
try:
    strict_bool07("sometimes")
except BoolConversionError07 as exc:
    assert isinstance(exc.__cause__, KeyError)
print("Problem 07 passed: exact type/value contract and case folding")


Problem 07 passed: exact type/value contract and case folding


**Review / production note:** Whether to accept whitespace or `yes/no` is an explicit interface decision, not an accidental side effect. Domain APIs may choose `TypeError` for unsupported types; this exercise intentionally exports one predictable domain exception.


---
## Problem 08 · Nested parsing: ignore unknown fields, reject invalid records

**Difficulty:** Advanced

### Challenge
From a JSON string mapping names to objects, create people with required valid ages. Ignore optional unknown fields, reject a record with missing or invalid age, and report each rejected record without aborting the entire batch. Make sure the valid records are not partially mutated by rejected input.

**Acceptance criteria:** The supplied assertions must pass without requiring network access or third-party libraries. Try solving it before opening the solution.


### Worked solution · reasoning
Treat one record as the unit of recovery. Validate required fields *before* committing a record to the result list. Use `KeyError` for absent fields, `ValueError` for invalid field values, and a narrow record-level catch. The original lesson uses `__slots__`/`setattr`; here allowlisting known fields avoids mass assignment from untrusted data.


In [16]:
@dataclass(frozen=True)
class Person08:
    name: str
    age: int


def decode_people08(text: str):
    payload = json.loads(text)  # malformed JSON aborts the whole document
    if not isinstance(payload, dict):
        raise TypeError("expected a JSON object mapping names to records")
    accepted, rejected = [], []
    for name, record in payload.items():
        try:
            if not isinstance(record, dict):
                raise ValueError("person must be an object")
            raw_age = record["age"]
            if isinstance(raw_age, bool) or not isinstance(raw_age, int) or raw_age < 0:
                raise ValueError("age must be a nonnegative integer")
            candidate = Person08(name=name, age=raw_age)
        except (KeyError, ValueError) as exc:
            rejected.append((name, type(exc).__name__, str(exc)))
        else:
            accepted.append(candidate)
    return accepted, rejected


### Verification · deterministic examples, boundary cases, and regression tests


In [17]:
sample_08 = json.dumps({
    "Alex": {"age": 18},
    "Bryan": {"age": 21, "city": "London"},
    "Guido": {"age": "unknown"},
    "Missing": {"city": "Sofia"},
    "Boolean": {"age": True},
})
valid_08, rejected_08 = decode_people08(sample_08)
assert valid_08 == [Person08("Alex", 18), Person08("Bryan", 21)]
assert [row[0] for row in rejected_08] == ["Guido", "Missing", "Boolean"]
try:
    decode_people08("{ definitely not json }")
except json.JSONDecodeError:
    pass
else:
    raise AssertionError("malformed document must propagate")
print("Problem 08 passed: per-record recovery and document-level failure")


Problem 08 passed: per-record recovery and document-level failure


**Review / production note:** Use schemas/allowlists for external records; catching `AttributeError` from arbitrary `setattr` can unintentionally conflate blocked fields with a property setter’s internal bug.


---
## Problem 09 · EAFP lookup that supports lists, mappings, and lengthless objects

**Difficulty:** Advanced

### Challenge
Implement `get_or_default09(container, key, default)` using direct subscripting rather than checking `len()`. Accept list indexes, dictionary keys, negative indexes, and an object exposing only `__getitem__`. Define whether an invalid key type should return the default; here it does.

**Acceptance criteria:** The supplied assertions must pass without requiring network access or third-party libraries. Try solving it before opening the solution.


### Worked solution · reasoning
The operation is `container[key]`; try that operation itself and intercept only its documented missing/unsupported-index failures (`IndexError`, `KeyError`, `TypeError`). Do not assume that every subscriptable object is sized, or that every key is numeric.


In [18]:
def get_or_default09(container, key, default=None):
    try:
        return container[key]
    except (IndexError, KeyError, TypeError):
        return default


class Endless09:
    def __getitem__(self, key):
        if not isinstance(key, int):
            raise TypeError("index must be integer")
        return 99


### Verification · deterministic examples, boundary cases, and regression tests


In [19]:
assert get_or_default09([5, 6], -1, "missing") == 6
assert get_or_default09([5, 6], 99, "missing") == "missing"
assert get_or_default09({"a": 100}, "a", "missing") == 100
assert get_or_default09({"a": 100}, "z", "missing") == "missing"
assert get_or_default09(Endless09(), 1_000_000, "missing") == 99
assert get_or_default09(Endless09(), "bad", "missing") == "missing"
assert get_or_default09(None, 0, "missing") == "missing"
print("Problem 09 passed: EAFP with lengthless sequence and mapping")


Problem 09 passed: EAFP with lengthless sequence and mapping


**Review / production note:** Catching `TypeError` may also hide a bug inside a custom `__getitem__`; narrow that policy for APIs where wrong key types should surface rather than return a default. EAFP is a tool, not a blanket rule.


---
## Problem 10 · Recoverable retries with injectable sleep

**Difficulty:** Expert

### Challenge
Implement `retry10(fn, attempts, sleep, delay)` for a flaky operation: retry **only** `TimeoutError`, sleep only between attempts, and propagate the final exception unchanged. Reject zero attempts. Make tests deterministic by using fake callables and a list as the clock.

**Acceptance criteria:** The supplied assertions must pass without requiring network access or third-party libraries. Try solving it before opening the solution.


### Worked solution · reasoning
Retry policy belongs at the boundary that understands whether an operation is safe to retry. Do not catch unrelated errors, do not sleep after final failure, and do not retry non-idempotent operations without explicit safeguards. Inject sleep rather than making tests slow or timing-dependent.


In [20]:
def retry10(fn, *, attempts: int, sleep, delay: float = 0.0):
    if isinstance(attempts, bool) or not isinstance(attempts, int) or attempts < 1:
        raise ValueError("attempts must be a positive integer")
    if delay < 0:
        raise ValueError("delay must be nonnegative")
    for index in range(attempts):
        try:
            return fn()
        except TimeoutError:
            if index == attempts - 1:
                raise
            sleep(delay)
    raise AssertionError("unreachable")


### Verification · deterministic examples, boundary cases, and regression tests


In [21]:
calls_10, sleeps_10 = [], []
def eventually_10():
    calls_10.append(1)
    if len(calls_10) < 3:
        raise TimeoutError("temporary")
    return "done"
assert retry10(eventually_10, attempts=4, sleep=sleeps_10.append, delay=0.25) == "done"
assert len(calls_10) == 3 and sleeps_10 == [0.25, 0.25]
final_10 = TimeoutError("last failure")
try:
    retry10(lambda: (_ for _ in ()).throw(final_10), attempts=2,
            sleep=sleeps_10.append, delay=1.0)
except TimeoutError as exc:
    assert exc is final_10
else:
    raise AssertionError("final failure must propagate unchanged")
assert sleeps_10 == [0.25, 0.25, 1.0]
try:
    retry10(lambda: (_ for _ in ()).throw(ValueError("no retry")),
            attempts=5, sleep=sleeps_10.append)
except ValueError:
    pass
else:
    raise AssertionError("unrelated error must propagate")
assert len(sleeps_10) == 3
for invalid in (0, -1, True, 2.5):
    try:
        retry10(lambda: None, attempts=invalid, sleep=lambda _: None)
    except ValueError:
        pass
    else:
        raise AssertionError("invalid attempts allowed")
print("Problem 10 passed: retry count, precise recovery, and final traceback")


Problem 10 passed: retry count, precise recovery, and final traceback


**Review / production note:** Production retry loops may additionally require exponential backoff, jitter, an overall deadline, cancellation awareness, and error observability. Decide idempotency before retrying writes.


---
## Problem 11 · Atomic in-memory transaction with rollback

**Difficulty:** Expert

### Challenge
Create a `Ledger11` with a balance and journal. Implement `transfer11(source, target, amount)` that commits both ledger updates or restores **both** on any failure. Model a deliberate post-debit failure through a target ledger’s `deposit` method. Preserve the original error after rollback.

**Acceptance criteria:** The supplied assertions must pass without requiring network access or third-party libraries. Try solving it before opening the solution.


### Worked solution · reasoning
Take snapshots before mutation and perform compensating rollback in `except`. The exception should remain visible, not turn into a false success. This toy example is single-threaded: real multi-writer transfers require a database transaction or locks.


In [22]:
class Ledger11:
    def __init__(self, balance: int, fail_deposit: bool = False):
        self.balance = balance
        self.journal = []
        self.fail_deposit = fail_deposit

    def debit(self, amount):
        if amount > self.balance:
            raise ValueError("insufficient funds")
        self.balance -= amount
        self.journal.append(("debit", amount))

    def deposit(self, amount):
        if self.fail_deposit:
            raise RuntimeError("credit service unavailable")
        self.balance += amount
        self.journal.append(("credit", amount))


def transfer11(source: Ledger11, target: Ledger11, amount: int):
    if type(amount) is not int or amount <= 0:
        raise ValueError("amount must be positive integer")
    source_snapshot = (source.balance, list(source.journal))
    target_snapshot = (target.balance, list(target.journal))
    try:
        source.debit(amount)
        target.deposit(amount)
    except Exception:
        source.balance, source.journal = source_snapshot
        target.balance, target.journal = target_snapshot
        raise


### Verification · deterministic examples, boundary cases, and regression tests


In [23]:
a_11, b_11 = Ledger11(100), Ledger11(40)
transfer11(a_11, b_11, 30)
assert (a_11.balance, b_11.balance) == (70, 70)
assert (a_11.journal, b_11.journal) == ([('debit', 30)], [('credit', 30)])
a_11, b_11 = Ledger11(100), Ledger11(40, fail_deposit=True)
try:
    transfer11(a_11, b_11, 25)
except RuntimeError as exc:
    assert str(exc) == "credit service unavailable"
else:
    raise AssertionError("deposit failure must propagate")
assert (a_11.balance, b_11.balance) == (100, 40)
assert a_11.journal == b_11.journal == []
try:
    transfer11(a_11, b_11, 1000)
except ValueError:
    pass
else:
    raise AssertionError("overdraft should fail")
assert (a_11.balance, b_11.balance) == (100, 40)
print("Problem 11 passed: all-or-nothing mutation")


Problem 11 passed: all-or-nothing mutation


**Review / production note:** This is a pedagogical rollback, not an ACID transaction: external side effects and concurrent modification cannot be reversed merely by restoring Python attributes.


---
## Problem 12 · Context manager: rollback with guaranteed restoration

**Difficulty:** Expert

### Challenge
Write a context manager that provides a mutable list as a temporary work area. If its body fails (including non-`Exception` exits), restore the original list contents; otherwise keep the changes. Do not suppress the original exception. Verify both exits.

**Acceptance criteria:** The supplied assertions must pass without requiring network access or third-party libraries. Try solving it before opening the solution.


### Worked solution · reasoning
A generator-based context manager can `yield` the resource once and use `finally` to restore it unless the body reached successful completion. Restoring via slice assignment keeps external references to the same list valid. Avoid hiding an exception by returning from a cleanup suite.


In [24]:
@contextlib.contextmanager
def tentative_edit12(items):
    snapshot = items.copy()
    completed = False
    try:
        yield items
        completed = True
    finally:
        if not completed:
            items[:] = snapshot


### Verification · deterministic examples, boundary cases, and regression tests


In [25]:
items_12 = ["original"]
with tentative_edit12(items_12) as work:
    work.append("committed")
assert items_12 == ["original", "committed"]
reference_12 = items_12
try:
    with tentative_edit12(items_12) as work:
        work.clear()
        work.append("temporary")
        raise RuntimeError("abort")
except RuntimeError as exc:
    assert str(exc) == "abort"
else:
    raise AssertionError("must re-raise")
assert items_12 is reference_12
assert items_12 == ["original", "committed"]
try:
    with tentative_edit12(items_12) as work:
        work.append("temporary")
        raise KeyboardInterrupt()
except KeyboardInterrupt:
    pass
else:
    raise AssertionError("interrupt must propagate")
assert items_12 == ["original", "committed"]
print("Problem 12 passed: commit, rollback, and BaseException-safe cleanup")


Problem 12 passed: commit, rollback, and BaseException-safe cleanup


**Review / production note:** This is safe for the list structure in a single-threaded example, not arbitrary referenced mutable objects inside it. A shallow copy will not undo mutations to nested objects.


---
## Problem 13 · ExitStack for partially acquired resources

**Difficulty:** Expert

### Challenge
Acquire an arbitrary sequence of context-managed resources. If the third acquisition fails, automatically close the first two in reverse order. If all acquisitions work, also release in reverse order. Ensure the failed object is not assumed to have been entered.

**Acceptance criteria:** The supplied assertions must pass without requiring network access or third-party libraries. Try solving it before opening the solution.


### Worked solution · reasoning
`contextlib.ExitStack` registers `__exit__` callbacks only after successful entry. Its LIFO unwind handles partial acquisition safely, with much less brittle nesting than multiple dynamic `with` statements.


In [26]:
class TrackedResource13:
    def __init__(self, name, events, fail_enter=False):
        self.name, self.events = name, events
        self.fail_enter = fail_enter
    def __enter__(self):
        self.events.append("enter:" + self.name)
        if self.fail_enter:
            raise OSError("cannot acquire " + self.name)
        return self
    def __exit__(self, exc_type, exc, tb):
        self.events.append("exit:" + self.name)
        return False


def acquire_all13(resources, events):
    with contextlib.ExitStack() as stack:
        acquired = [stack.enter_context(resource) for resource in resources]
        events.append("work:" + ",".join(r.name for r in acquired))
    return len(acquired)


### Verification · deterministic examples, boundary cases, and regression tests


In [27]:
events_13 = []
resources_13 = [TrackedResource13(n, events_13) for n in ("A", "B", "C")]
assert acquire_all13(resources_13, events_13) == 3
assert events_13 == ["enter:A", "enter:B", "enter:C", "work:A,B,C",
                     "exit:C", "exit:B", "exit:A"]
events_13 = []
resources_13 = [TrackedResource13("A", events_13), TrackedResource13("B", events_13),
                TrackedResource13("C", events_13, fail_enter=True)]
try:
    acquire_all13(resources_13, events_13)
except OSError as exc:
    assert "C" in str(exc)
else:
    raise AssertionError("third resource must fail")
assert events_13 == ["enter:A", "enter:B", "enter:C", "exit:B", "exit:A"]
print("Problem 13 passed: deterministic LIFO cleanup after partial acquisition")


Problem 13 passed: deterministic LIFO cleanup after partial acquisition


**Review / production note:** `__exit__` should return a truthy value only when deliberate suppression is part of the resource’s contract. Prefer `ExitStack` to manually juggling a variable number of `finally` clauses.


---
## Problem 14 · Atomic JSON output with failure injection

**Difficulty:** Expert

### Challenge
Write JSON to a temporary file in the destination directory, then replace the destination with `os.replace`. If serialization or replacement fails, remove the temporary file and preserve the original destination. Inject a replacement function to simulate a failure. Use only a temporary directory in tests.

**Acceptance criteria:** The supplied assertions must pass without requiring network access or third-party libraries. Try solving it before opening the solution.


### Worked solution · reasoning
A same-directory temp file plus `os.replace` provides atomic name replacement on filesystems that support it. Closing and flushing the file precede replace. `fsync` on the temp file improves write persistence, but complete crash durability can additionally need a directory sync and is platform-dependent. Use `finally` to delete an uncommitted temp path.


In [28]:
def write_json_atomic14(destination, payload, *, replace_fn=os.replace):
    destination = Path(destination)
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w", encoding="utf-8", dir=destination.parent,
            prefix=f".{destination.name}.", suffix=".tmp", delete=False
        ) as handle:
            temporary = Path(handle.name)
            json.dump(payload, handle, ensure_ascii=False, sort_keys=True)
            handle.write("\n")
            handle.flush()
            os.fsync(handle.fileno())
        replace_fn(temporary, destination)
        temporary = None  # replacement succeeded: this temp pathname no longer exists
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)


### Verification · deterministic examples, boundary cases, and regression tests


In [29]:
with tempfile.TemporaryDirectory() as directory_14:
    target_14 = Path(directory_14) / "state.json"
    target_14.write_text('OLD CONTENT', encoding="utf-8")
    try:
        write_json_atomic14(target_14, {"bad": {1, 2}})
    except TypeError:
        pass
    else:
        raise AssertionError("sets cannot be JSON serialized")
    assert target_14.read_text(encoding="utf-8") == 'OLD CONTENT'
    assert sorted(p.name for p in target_14.parent.iterdir()) == ["state.json"]

    def fail_replace14(source, destination):
        assert Path(source).exists()
        raise OSError("injected rename failure")

    try:
        write_json_atomic14(target_14, {"ok": 1}, replace_fn=fail_replace14)
    except OSError as exc:
        assert "injected" in str(exc)
    else:
        raise AssertionError("injected error must propagate")
    assert target_14.read_text(encoding="utf-8") == 'OLD CONTENT'
    assert sorted(p.name for p in target_14.parent.iterdir()) == ["state.json"]

    write_json_atomic14(target_14, {"name": "Sofia", "ok": 1})
    assert json.loads(target_14.read_text(encoding="utf-8")) == {"name": "Sofia", "ok": 1}
print("Problem 14 passed: old file preserved and orphaned temp files removed")


Problem 14 passed: old file preserved and orphaned temp files removed


**Review / production note:** Replacement atomicity is not identical to all-platform crash durability. Use filesystem guarantees appropriate to your environment; cross-device replaces may fail, which is why the temp file is placed in the target directory.


---
## Problem 15 · SQLite savepoint: catch, roll back, and continue

**Difficulty:** Expert

### Challenge
Use an in-memory SQLite database with a UNIQUE constraint. Implement `insert_batch15(conn, names)` inside a SAVEPOINT; on an integrity error, roll back only that batch and re-raise. Verify that preexisting rows survive and that a later valid batch can still commit.

**Acceptance criteria:** The supplied assertions must pass without requiring network access or third-party libraries. Try solving it before opening the solution.


### Worked solution · reasoning
Database transactions, not Python object snapshots, protect persisted records. SAVEPOINT allows a partial transaction rollback. SQL savepoint identifiers cannot be value-bound with `?`, so use a constant identifier owned by this function; real reusable libraries should manage unique savepoint names.


In [30]:
def insert_batch15(connection, names):
    connection.execute("SAVEPOINT import_batch_15")
    try:
        for name in names:
            connection.execute("INSERT INTO users_15 (name) VALUES (?)", (name,))
    except sqlite3.Error:
        connection.execute("ROLLBACK TO SAVEPOINT import_batch_15")
        connection.execute("RELEASE SAVEPOINT import_batch_15")
        raise
    else:
        connection.execute("RELEASE SAVEPOINT import_batch_15")


### Verification · deterministic examples, boundary cases, and regression tests


In [31]:
with contextlib.closing(sqlite3.connect(":memory:")) as conn_15:
    conn_15.execute("CREATE TABLE users_15 (name TEXT NOT NULL UNIQUE)")
    conn_15.execute("INSERT INTO users_15 (name) VALUES (?)", ("existing",))
    conn_15.commit()
    with conn_15:
        try:
            insert_batch15(conn_15, ["new", "existing", "never"])
        except sqlite3.IntegrityError:
            pass
        else:
            raise AssertionError("duplicate name must fail")
        assert conn_15.execute("SELECT name FROM users_15 ORDER BY name").fetchall() == [("existing",)]
        insert_batch15(conn_15, ["Alice", "Bob"])
    assert conn_15.execute("SELECT name FROM users_15 ORDER BY name").fetchall() == [
        ("Alice",), ("Bob",), ("existing",)
    ]
print("Problem 15 passed: savepoint rollback and subsequent successful batch")


Problem 15 passed: savepoint rollback and subsequent successful batch


**Review / production note:** Do not silently catch every database failure and continue: distinguish uniqueness violations from a lost connection or corrupted database. `with sqlite3.Connection` manages transaction completion, but it does **not** close the connection.


---
## Problem 16 · Diagnostic logging without swallowing or leaking secrets

**Difficulty:** Expert

### Challenge
Write an adapter that parses a numeric configuration value and converts a `ValueError` into `ConfigurationError`. Log the record identifier and traceback, but NEVER include the raw secret token in the log or public message. Capture logging output in tests without touching the root logger.

**Acceptance criteria:** The supplied assertions must pass without requiring network access or third-party libraries. Try solving it before opening the solution.


### Worked solution · reasoning
Log once at a useful system boundary with `logger.exception()` while handling the error; it automatically attaches the active exception traceback. Translate with `from exc`. Beware that a traceback prints the *original exception message*: `int(secret)` currently may include the secret, so validate via a safe pattern or explicitly suppress sensitive causes when logging. Here, precheck ASCII digits and raise a sanitized internal `ValueError` before logging.


In [32]:
class SecretConfigurationError16(Exception):
    pass


def parse_secret_code16(token, record_id, logger):
    try:
        if not isinstance(token, str) or not token.isascii() or not token.isdigit():
            raise ValueError("secret code must contain ASCII decimal digits")
        if len(token) > 4:
            raise ValueError("secret code must be at most four digits")
        value = int(token)
        if value > 9999:
            raise ValueError("secret code out of supported range")
    except ValueError as exc:
        logger.exception("record %s has an invalid secret code", record_id)
        raise SecretConfigurationError16(f"record {record_id}: invalid secret code") from exc
    return value


### Verification · deterministic examples, boundary cases, and regression tests


In [33]:
stream_16 = io.StringIO()
logger_16 = logging.getLogger("advanced_exception_notebook_16")
logger_16.handlers.clear()
logger_16.propagate = False
logger_16.setLevel(logging.DEBUG)
handler_16 = logging.StreamHandler(stream_16)
handler_16.setFormatter(logging.Formatter("%(levelname)s:%(message)s"))
logger_16.addHandler(handler_16)
try:
    assert parse_secret_code16("0007", "row-2", logger_16) == 7
    try:
        parse_secret_code16("TOP_SECRET_PW", "row-3", logger_16)
    except SecretConfigurationError16 as exc:
        assert isinstance(exc.__cause__, ValueError)
        assert "TOP_SECRET_PW" not in str(exc)
    else:
        raise AssertionError("bad secret must fail")
    content_16 = stream_16.getvalue()
    assert "row-3" in content_16 and "ValueError" in content_16
    assert "TOP_SECRET_PW" not in content_16
    # Long digit strings are screened before int() can generate an input-bearing error.
    try:
        parse_secret_code16("9" * 5000, "row-4", logger_16)
    except SecretConfigurationError16:
        pass
    else:
        raise AssertionError("oversized input must fail safely")
    assert "999999999999999" not in stream_16.getvalue()
finally:
    logger_16.removeHandler(handler_16)
    handler_16.close()
print("Problem 16 passed: traceback retained but secret token not logged")


Problem 16 passed: traceback retained but secret token not logged


**Review / production note:** For truly sensitive tokens, use parsing libraries and exception types that do not embed original values in their messages, or avoid logging the original cause altogether. Never assume `logger.exception()` itself redacts traceback data.


---
## Problem 17 · Narrow, intentional suppression

**Difficulty:** Advanced

### Challenge
Implement removal of a file that should be considered successful if the file was already missing. Suppress only `FileNotFoundError`, not `PermissionError` or other `OSError` subclasses. Demonstrate that `contextlib.suppress` documents the policy clearly.

**Acceptance criteria:** The supplied assertions must pass without requiring network access or third-party libraries. Try solving it before opening the solution.


### Worked solution · reasoning
A missing file is an expected race during cleanup; permission errors are not. Use `contextlib.suppress(FileNotFoundError)` around only the operation whose absence is acceptable.


In [34]:
def remove_if_present17(path):
    with contextlib.suppress(FileNotFoundError):
        path.unlink()


class ForbiddenPath17:
    def unlink(self):
        raise PermissionError("access denied")


### Verification · deterministic examples, boundary cases, and regression tests


In [35]:
with tempfile.TemporaryDirectory() as directory_17:
    file_17 = Path(directory_17) / "transient.tmp"
    file_17.write_text("temporary", encoding="utf-8")
    remove_if_present17(file_17)
    assert not file_17.exists()
    remove_if_present17(file_17)  # already absent is fine
try:
    remove_if_present17(ForbiddenPath17())
except PermissionError as exc:
    assert str(exc) == "access denied"
else:
    raise AssertionError("permission error must not be suppressed")
print("Problem 17 passed: precise suppression without masking unrelated errors")


Problem 17 passed: precise suppression without masking unrelated errors


**Review / production note:** Do not use `except OSError: pass` merely to make cleanup appear robust: it can conceal permission, disk, and filesystem failures that warrant attention.


---
## Problem 18 · Generators, `close()`, and deterministic cleanup

**Difficulty:** Expert

### Challenge
Build a generator that yields one item and then raises `ValueError`. It must append a cleanup event in `finally` when closed early or when iteration raises. Demonstrate that `with contextlib.closing(generator)` closes the generator on early exit; a bare `break` alone is not a portable guarantee of immediate generator finalization.

**Acceptance criteria:** The supplied assertions must pass without requiring network access or third-party libraries. Try solving it before opening the solution.


### Worked solution · reasoning
A generator is suspended at `yield`. Calling `.close()` on a started generator triggers `GeneratorExit` at its suspension point, unwinding its `finally`. `contextlib.closing` makes the lifetime explicit, independent of implementation-specific garbage collection.


In [36]:
def stream18(events):
    try:
        events.append("start")
        yield "first"
        events.append("resume")
        raise ValueError("stream decode failed")
    finally:
        events.append("cleanup")


### Verification · deterministic examples, boundary cases, and regression tests


In [37]:
events_18 = []
with contextlib.closing(stream18(events_18)) as source_18:
    assert next(source_18) == "first"
assert events_18 == ["start", "cleanup"]
events_18 = []
with contextlib.closing(stream18(events_18)) as source_18:
    assert next(source_18) == "first"
    try:
        next(source_18)
    except ValueError as exc:
        assert str(exc) == "stream decode failed"
    else:
        raise AssertionError("stream error did not propagate")
assert events_18 == ["start", "resume", "cleanup"]
print("Problem 18 passed: explicit generator close and failure cleanup")


Problem 18 passed: explicit generator close and failure cleanup


**Review / production note:** Never yield again while handling `GeneratorExit`; that causes `RuntimeError: generator ignored GeneratorExit`. As with other cleanup, avoid replacing the primary exception with an accidental cleanup error.


---
## Problem 19 · ExceptionGroup and partial handling with except*

**Difficulty:** Expert · Python 3.11+

### Challenge
Process a group containing `ValueError`, `KeyError`, and `RuntimeError`. Collect the value/key messages separately with `except*`, then verify the unhandled runtime subgroup still propagates. Preserve errors instead of flattening them into strings too early.

**Acceptance criteria:** The supplied assertions must pass without requiring network access or third-party libraries. Try solving it before opening the solution.


### Worked solution · reasoning
`except*` splits exception groups by exception type; multiple matching clauses can each run on different subgroups. The unmatched subgroup is re-raised automatically. An `except*` clause cannot use `return`, `break`, or `continue`, so record observations and let the outer caller handle survivors.


In [38]:
def partition_group19(group, events):
    try:
        raise group
    except* ValueError as subgroup:
        events.extend(("value", str(e)) for e in subgroup.exceptions)
    except* KeyError as subgroup:
        events.extend(("key", e.args[0]) for e in subgroup.exceptions)


### Verification · deterministic examples, boundary cases, and regression tests


In [39]:
events_19 = []
group_19 = ExceptionGroup("mixed failures", [
    ValueError("invalid age"), KeyError("name"), RuntimeError("service down")
])
try:
    partition_group19(group_19, events_19)
except ExceptionGroup as remainder_19:
    assert len(remainder_19.exceptions) == 1
    assert isinstance(remainder_19.exceptions[0], RuntimeError)
    assert str(remainder_19.exceptions[0]) == "service down"
else:
    raise AssertionError("unhandled subgroup must propagate")
assert events_19 == [("value", "invalid age"), ("key", "name")]
print("Problem 19 passed: except* partitions and propagates unmatched errors")


Problem 19 passed: except* partitions and propagates unmatched errors


**Review / production note:** `except*` catches matching leaves of nested groups and preserves group structure as needed; do not assume every `.exceptions` element is always a simple exception for arbitrary nested inputs.


---
## Problem 20 · Async task cancellation must still clean up

**Difficulty:** Expert

### Challenge
Run a task that blocks on an await, cancel it deterministically, and prove its `finally` cleanup executes. The coordinator should acknowledge `asyncio.CancelledError` without converting cancellation to a successful result. Do not use real multi-second sleeps or external services.

**Acceptance criteria:** The supplied assertions must pass without requiring network access or third-party libraries. Try solving it before opening the solution.


### Worked solution · reasoning
Cancellation is injected at a suspension point. `asyncio.CancelledError` inherits `BaseException` on modern Python; broad `except Exception` does not catch it. Keep cleanup in `finally`, and handle cancellation only at a boundary that deliberately awaits the cancelled task.


In [40]:
async def cancellable_worker20(events):
    try:
        events.append("started")
        await asyncio.sleep(3600)  # cancelled before the sleep finishes
    finally:
        events.append("cleaned")


async def demonstrate_cancellation20():
    events = []
    task = asyncio.create_task(cancellable_worker20(events))
    await asyncio.sleep(0)  # let worker reach its first await
    task.cancel()
    try:
        await task
    except asyncio.CancelledError:
        events.append("cancel-observed")
    else:
        raise AssertionError("task should have been cancelled")
    assert task.cancelled()
    return events


### Verification · deterministic examples, boundary cases, and regression tests


In [41]:
events_20 = await demonstrate_cancellation20()
assert events_20 == ["started", "cleaned", "cancel-observed"]
assert not issubclass(asyncio.CancelledError, Exception)
print("Problem 20 passed: cancellation preserves finally cleanup")


Problem 20 passed: cancellation preserves finally cleanup


**Review / production note:** Most application coroutines should let cancellation propagate after cleanup. If you intercept cancellation and intend to suppress it, understand structured-concurrency and task cancellation state first.


---
## Problem 21 · Aggregate structured validation failures

**Difficulty:** Expert · Python 3.11+

### Challenge
Validate an object with required `host` (nonempty string), `port` (int 1–65535, rejecting bool), and `timeout` (positive int). Report *every* invalid field in one `ExceptionGroup`. Make each contained exception record the field and attach a note describing the expected contract.

**Acceptance criteria:** The supplied assertions must pass without requiring network access or third-party libraries. Try solving it before opening the solution.


### Worked solution · reasoning
Collect independent validation failures before raising one group. Keep structured fields for programs, and human-readable notes for debugging. Return a normalized dictionary only if all checks succeed. A flat group is sufficient here; more complex systems might group errors by record.


In [42]:
class FieldIssue21(ValueError):
    def __init__(self, field, message):
        self.field = field
        super().__init__(f"{field}: {message}")


def validate_config21(payload):
    if not isinstance(payload, dict):
        raise TypeError("configuration must be a dict")
    errors = []
    rules = {
        "host": lambda x: isinstance(x, str) and bool(x.strip()),
        "port": lambda x: type(x) is int and 1 <= x <= 65535,
        "timeout": lambda x: type(x) is int and x > 0,
    }
    expectations = {
        "host": "nonempty string", "port": "integer 1..65535, not bool",
        "timeout": "positive integer, not bool",
    }
    for field, check in rules.items():
        if field not in payload or not check(payload[field]):
            issue = FieldIssue21(field, "missing or invalid")
            issue.add_note("expected " + expectations[field])
            errors.append(issue)
    if errors:
        raise ExceptionGroup("configuration validation failed", errors)
    return {field: payload[field] for field in rules}


### Verification · deterministic examples, boundary cases, and regression tests


In [43]:
assert validate_config21({"host": "example", "port": 443, "timeout": 10}) == {
    "host": "example", "port": 443, "timeout": 10
}
try:
    validate_config21({"host": "  ", "port": True, "timeout": -2})
except ExceptionGroup as group_21:
    assert len(group_21.exceptions) == 3
    assert {e.field for e in group_21.exceptions} == {"host", "port", "timeout"}
    assert all(e.__notes__ and e.__notes__[0].startswith("expected")
               for e in group_21.exceptions)
else:
    raise AssertionError("all three errors must be collected")
try:
    validate_config21({"host": "valid", "port": 443})
except ExceptionGroup as group_21:
    assert [e.field for e in group_21.exceptions] == ["timeout"]
print("Problem 21 passed: field-level errors and human-readable notes")


Problem 21 passed: field-level errors and human-readable notes


**Review / production note:** `ExceptionGroup` is suited to truly independent concurrent or batch failures; for a simple form, a returned list of issues may be easier for callers. Prefer explicit API contracts over aggregation for its own sake.


---
## Problem 22 · Capstone: validate every row, then atomically export

**Difficulty:** Expert · integration

### Challenge
Build a small import pipeline for a JSON array of order records. Every row requires a unique nonempty string `id`, a strictly positive integer `quantity` (reject bool), and an `active` token parsed by Problem 07. Accumulate all invalid rows into an `ExceptionGroup` of structured `RowIssue22` exceptions; write **nothing** if any row is bad. On complete success, atomically replace the output using Problem 14. Simulate a failed replace and verify the old output survives. No network, external dataset, or installed packages are needed.

**Acceptance criteria:** The supplied assertions must pass without requiring network access or third-party libraries. Try solving it before opening the solution.


### Worked solution · reasoning
Split the pipeline into parse → validate → commit. Parsing errors are document-level failures. Row failures contain index/field/cause and aggregate without partial write. Write through the previously tested atomic function only after validation succeeds. This capstone deliberately composes the earlier `strict_bool07` and `write_json_atomic14` implementations.


In [44]:
class RowIssue22(ValueError):
    def __init__(self, row, field, message):
        self.row, self.field = row, field
        super().__init__(f"row {row}, {field}: {message}")


def parse_order22(record, index, used_ids):
    if not isinstance(record, dict):
        raise RowIssue22(index, "record", "expected an object")
    try:
        ident = record["id"]
    except KeyError as exc:
        raise RowIssue22(index, "id", "missing") from exc
    if not isinstance(ident, str) or not ident.strip():
        raise RowIssue22(index, "id", "must be a nonempty string")
    if ident in used_ids:
        raise RowIssue22(index, "id", "duplicate id")
    try:
        quantity = record["quantity"]
    except KeyError as exc:
        raise RowIssue22(index, "quantity", "missing") from exc
    if type(quantity) is not int or quantity <= 0:
        raise RowIssue22(index, "quantity", "must be a positive integer")
    try:
        active = strict_bool07(record["active"])
    except KeyError as exc:
        raise RowIssue22(index, "active", "missing") from exc
    except BoolConversionError07 as exc:
        raise RowIssue22(index, "active", "invalid Boolean token") from exc
    # Callers reserve the identifier only AFTER a completely valid record.
    return {"id": ident, "quantity": quantity, "active": active}


def export_orders22(raw_json, destination, *, replace_fn=os.replace):
    payload = json.loads(raw_json)
    if not isinstance(payload, list):
        raise TypeError("orders document must be a JSON array")
    normalized, errors, used_ids = [], [], set()
    for index, record in enumerate(payload, start=1):
        try:
            clean = parse_order22(record, index, used_ids)
        except RowIssue22 as exc:
            errors.append(exc)
        else:
            used_ids.add(clean["id"])
            normalized.append(clean)
    if errors:
        raise ExceptionGroup("invalid order rows; output not modified", errors)
    write_json_atomic14(destination, normalized, replace_fn=replace_fn)
    return len(normalized)


### Verification · deterministic examples, boundary cases, and regression tests


In [45]:
with tempfile.TemporaryDirectory() as directory_22:
    output_22 = Path(directory_22) / "orders.json"
    output_22.write_text('ORIGINAL', encoding="utf-8")
    broken_22 = json.dumps([
        {"id": "A", "quantity": 2, "active": "TRUE"},
        {"id": "B", "quantity": True, "active": "false"},
        {"id": "C", "quantity": 1, "active": "maybe"},
        {"id": "A", "quantity": 3, "active": 1},
        {"id": "D", "quantity": 0, "active": "t"},
        {"id": "E", "quantity": 1},
    ])
    try:
        export_orders22(broken_22, output_22)
    except ExceptionGroup as group_22:
        assert len(group_22.exceptions) == 5
        assert [(e.row, e.field) for e in group_22.exceptions] == [
            (2, "quantity"), (3, "active"), (4, "id"),
            (5, "quantity"), (6, "active")
        ]
    else:
        raise AssertionError("bad batch must fail atomically")
    assert output_22.read_text(encoding="utf-8") == 'ORIGINAL'

    good_22 = json.dumps([
        {"id": "A", "quantity": 2, "active": "TRUE", "ignored": 999},
        {"id": "B", "quantity": 3, "active": "false"},
    ])
    def reject_replace22(source, destination):
        raise OSError("simulated disk rename failure")
    try:
        export_orders22(good_22, output_22, replace_fn=reject_replace22)
    except OSError as exc:
        assert "rename" in str(exc)
    else:
        raise AssertionError("replace failure must escape")
    assert output_22.read_text(encoding="utf-8") == 'ORIGINAL'
    assert sorted(p.name for p in output_22.parent.iterdir()) == ["orders.json"]

    assert export_orders22(good_22, output_22) == 2
    assert json.loads(output_22.read_text(encoding="utf-8")) == [
        {"active": True, "id": "A", "quantity": 2},
        {"active": False, "id": "B", "quantity": 3},
    ]
    # Duplicate IDs are reported when an earlier valid row already claimed the ID.
    try:
        export_orders22(json.dumps([
            {"id": "Z", "quantity": 1, "active": True},
            {"id": "Z", "quantity": 2, "active": False},
        ]), output_22)
    except ExceptionGroup as group_22:
        assert len(group_22.exceptions) == 1
        assert (group_22.exceptions[0].row, group_22.exceptions[0].field) == (2, "id")
    else:
        raise AssertionError("duplicate id must fail")
    assert len(json.loads(output_22.read_text(encoding="utf-8"))) == 2
print("Problem 22 passed: no partial export, aggregated diagnostics, atomic publish")


Problem 22 passed: no partial export, aggregated diagnostics, atomic publish


**Review / production note:** This capstone’s uniqueness policy reserves only fully valid rows; document it if business rules instead require reporting duplicate IDs even when the first occurrence is invalid. For larger files, consider streaming validation plus bounded temporary storage and well-defined limits.


---
# Final mastery checklist and extension challenges

You should now be able to explain which `except` clause matches, why `else` differs from code after a handled exception, when `finally` can accidentally mask a failure, why `raise ... from exc` is useful, how to choose an exception hierarchy, when EAFP is preferable to pre-checks, how to constrain retries, and how to keep cleanup and transactional writes correct.

**Extra challenges (not required for the executable answer key):** add jitter and a deadline to Problem 10; add a durable directory sync strategy for your platform to Problem 14; generalize Problem 15 to nested savepoints with unique names; add bounded error-count reporting to Problem 22; and test the capstone with property-based testing if your environment permits third-party packages.

**Run-all criterion:** Every verification cell prints `Problem NN passed`, with no uncaught exceptions. The notebook uses temporary directories and an in-memory database, leaving no permanent test data behind.

**Source relationship:** The uploaded lesson covers fundamental handlers, `finally`, `else`, nested parsing, custom Boolean conversion, and EAFP/permission-first approaches. This notebook preserves those foundations, corrects editorial/terminology mistakes, and adds new production-oriented scenarios and modern Python 3.11+ features.


In [46]:
# Lightweight final self-check: main APIs from the integrated problems remain callable.
assert strict_bool07("FALSE") is False
assert classify_error(MalformedField("bad")) == "malformed-field"
assert get_or_default09({"x": 10}, "x") == 10
assert validate_config21({"host": "h", "port": 80, "timeout": 1})["port"] == 80
print("All 22 problem verification cells have run successfully if you see this after Run All.")


All 22 problem verification cells have run successfully if you see this after Run All.
